# TPS Regulator Calcs

This Jupyter Notebook allows you to input various design goals for the
TPS7A4901 adjustable LDO regulator and calculate recommended resistor-divider
values, power dissipation, startup time, etc. The equations are based on
Texas Instruments’ datasheet discussion and the example forum post.

In [32]:
import math

In [ ]:
# -------------------------------------------------------------------------------------
#                      USER-EDITABLE INPUTS
# -------------------------------------------------------------------------------------

# Design Goals
VIN       = 3.0       # Input voltage to the LDO (V)
VOUT      = 1.2       # Desired output voltage (V)
IOUT      = 0.150     # Load current (A)

# LDO Parameters
TA_max    = 75.0      # Maximum ambient temperature (°C)
VDO_max   = 0.600     # Worst-case dropout voltage (V) from datasheet at IOUT=150mA
Iq        = 0.0010    # Quiescent current (A) typical or worst-case from datasheet
Vfb_nom   = 1.185     # Nominal feedback voltage (V)
Vfb_min   = 1.18      # Minimum feedback voltage (V) for margin checks
Vfb_max   = 1.195     # Maximum feedback voltage (V) for margin checks
I_div_min = 5e-6      # Minimum recommended resistor divider current (A)
RnJA      = 63.5      # Junction-to-ambient thermal resistance (°C/W), example for DGN pkg


R2_choice = 100e3     # Chosen R2 (Ω), 100kΩ 1% tolerance; datasheet: max 237kΩ
Cnr_ss    = 10e-9     # Soft-start capacitor (F), 10nF X7R
Cout      = 10e-6     # Output capacitor (F), 10uF X7R
Cin       = 10e-6     # Input capacitor (F), 10uF X7R
# 
# If your datasheet or chosen package has different RθJA, update RnJA above.

In [34]:
# 1) Check dropout headroom
dropout_headroom = VIN - VOUT - VDO_max
print(f"1) Dropout headroom = VIN - VOUT - VDO_max = {dropout_headroom:.3f} V")
if dropout_headroom < 1.0: # Datasheet: for optimal performance must be at least 1 V
    print("   *Warning*: dropout headroom < 1 V. Ensure dropout margin is adequate.\n")
else:
    print("   Dropout headroom >= 1.0 V (sufficient).\n")

1) Dropout headroom = VIN - VOUT - VDO_max = 1.200 V
   Dropout headroom >= 1.0 V (sufficient).



In [35]:
# 2) Check resistor network for no-load stability (must exceed 5 μA)
I_div_actual = Vfb_nom / R2_choice #    Datasheet: I_divider = Vfb / R2 > 5 μA

print(f"2) Divider current I_div = {I_div_actual*1e6:.3f} μA with R2 = {R2_choice/1e3:.1f} kΩ")
if I_div_actual < I_div_min:
    print("   *Warning*: Divider current is below the 5 μA recommended minimum!\n")
else:
    print("   Divider current is above 5 μA, which is good for stability.\n")

2) Divider current I_div = 11.850 μA with R2 = 100.0 kΩ
   Divider current is above 5 μA, which is good for stability.



In [36]:
# 3) Solve for R1 using VOUT = Vfb * (1 + R1/R2)
#    => R1 = R2 * [(VOUT / Vfb) - 1]
R1_calc_nom = R2_choice * ((VOUT / Vfb_nom) - 1.0)

# Some corner checks with Vfb_min, Vfb_max:
R1_calc_min = R2_choice * ((VOUT / Vfb_max) - 1.0)
R1_calc_max = R2_choice * ((VOUT / Vfb_min) - 1.0)

print("3) Resistor R1 Calculation (nominal feedback voltage)")
print(f"   R1 (nominal) = {R1_calc_nom/1e3:.3f} kΩ")

# Also show the min and max possible R1 for feedback tolerance
print("   Considering Vfb tolerance:")
print(f"   - R1 (with Vfb_max={Vfb_max:.3f} V) = {R1_calc_min/1e3:.3f} kΩ")
print(f"   - R1 (with Vfb_min={Vfb_min:.3f} V) = {R1_calc_max/1e3:.3f} kΩ")
if R1_calc_nom < 0:
    print("   *Warning*: R1 is negative => chosen R2 is too large for your target VOUT.\n")

3) Resistor R1 Calculation (nominal feedback voltage)
   R1 (nominal) = 1.266 kΩ
   Considering Vfb tolerance:
   - R1 (with Vfb_max=1.195 V) = 0.418 kΩ
   - R1 (with Vfb_min=1.180 V) = 1.695 kΩ


In [37]:
# 4) Maximum power dissipation in the LDO:
#    PD = (VIN - VOUT) * IOUT + VIN * Iq
#    Datasheet curves show 1.2-1.6W is the typical upper end of power dissipation for this LDO.
Pd = (VIN - VOUT)*IOUT + VIN*Iq
print(f"\n4) Power dissipation, Pd = (VIN - VOUT)*IOUT + VIN*Iq = {Pd*1e3:.3f} mW")


4) Power dissipation, Pd = (VIN - VOUT)*IOUT + VIN*Iq = 273.000 mW


In [38]:
# 5) Junction temperature rise:
#    Tj_rise = Pd * RθJA
Tj_rise = Pd * RnJA
Tj_max = TA_max + Tj_rise
print(f"   Junction temp rise = Pd * RθJA = {Tj_rise:.2f} °C above ambient")
print(f"   => Tj(max) at TA={TA_max:.1f}°C => {Tj_max:.2f} °C")
#    We want to keep Tj(max) comfortably below device max rating (e.g., 125°C)
if Tj_max > 125.0:
    print("   *Warning*: Junction temperature exceeds 125°C limit!\n")
else:
    print("   Tj(max) is below typical 125°C limit (check your datasheet for exact max).\n")

   Junction temp rise = Pd * RθJA = 17.34 °C above ambient
   => Tj(max) at TA=75.0°C => 92.34 °C
   Tj(max) is below typical 125°C limit (check your datasheet for exact max).



In [39]:
# 6) Approximate startup time with the soft-start capacitor:
#    For TPS7A49, a simplified example from the datasheet uses ~1.4 * CNR/SS to get nominal tSS:
tSS_est = 1.4 * (Cnr_ss / 10e-9)
print(f"7) Estimated startup time (soft-start dominated) ≈ {tSS_est:.3f} ms\n")

7) Estimated startup time (soft-start dominated) ≈ 1.400 ms



In [40]:
# 7) Soft-start capacitor values
Cin_soft = 2.2e-6 # 2.2 μF for soft-start
Cout_soft_min = Cin_soft
ICL_max = 0.5 # Datasheet: 500 mA max load current for soft-start, worst-case maximum short-circuit or startup current limit
tss_cl = VOUT * (Cout_soft_min / ICL_max)
Cout_soft_max = tss_cl * (ICL_max / VOUT) # max capacitance for soft-start

In [ ]:
# 8) Recommended capacitor selection
print("8) Capacitor Recommendations:")
print(f"   - Input capacitor CIN = {Cin*1e6:.2f} μF")
print(f"   - Output capacitor COUT = {Cout*1e6:.2f} μF")
print(f"   - Soft-start capacitor CNR/SS = {Cnr_ss*1e9:.2f} nF")
print(f"   - R1 = {R1_calc_nom/1e3:.3f} kΩ (nominal)")
print(f"   - R2 = {R2_choice/1e3:.3f} kΩ (chosen)\n")

# Soft-start
print(f"   - Soft-start COUT(min) = {Cout_soft_min*1e6:.2f} μF (min for soft-start)")
print(f"   - Soft-start COUT(max) = {Cout_soft_max*1e6:.2f} μF (max for soft-start)\n")

print("For improved PSRR, use 10 μF input and output capacitors. To reduce transient peaks (at the cost of slower recovery), increase or add output capacitors (soft start).\n")

print("Done.\n")

8) Capacitor Recommendations:
   - Input capacitor CIN = 10.00 μF
   - Output capacitor COUT = 10.00 μF
   - Soft-start capacitor CNR/SS = 10.00 nF
   - R1 = 1.266 kΩ (nominal)
   - R2 = 100.000 kΩ (chosen)

   - Soft-start COUT(min) = 2.20 μF (min for soft-start)
   - Soft-start COUT(max) = 2.20 μF (max for soft-start)

For improved PSRR, use 10 μF input and output capacitors. To reduce transient peaks (at the cost of slower recovery), increase or add output capacitors.

Done.

